# Pelajaran 09 - Corak Reka Bentuk Metakognisi


## Persediaan

Notebook ini menunjukkan corak reka bentuk Metakognisi menggunakan Microsoft Agent Framework.

**Prasyarat:**
- Penyebaran Azure OpenAI dikonfigurasikan melalui pembolehubah persekitaran
- Azure CLI diautentikasi (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apakah Metakognisi?

Metakognisi adalah **berfikir tentang berfikir**. Dalam konteks ejen AI, ia bermaksud membina ejen yang boleh:

- **Melakukan refleksi diri** terhadap keluaran dan proses penalaran mereka sendiri
- **Mengesan ralat** dan memulihkan diri dengan baik daripada gagal secara senyap
- **Menilai** sama ada respons mereka lengkap dan membantu
- **Menyesuaikan** strategi mereka apabila pendekatan awal tidak berhasil (contohnya, beralih kepada sistem sandaran)

Ejen metakognitif bukan sekadar menjawab soalan — ia memantau prestasinya sendiri dan menyesuaikan diri secara serta-merta.


## Alat Utama dan Sandaran

Satu corak metakognisi yang biasa ialah **strategi sandaran**. Ejen mencuba alat utama terlebih dahulu; jika ia gagal (contohnya, ralat 404), ejen mengenal pasti kegagalan itu dan dengan telus beralih kepada alat sandaran.

Ini mencerminkan sistem dunia nyata di mana perkhidmatan utama mungkin tidak tersedia dan ejen mesti mendiagnosis sendiri isu tersebut sebelum memilih laluan alternatif.

Di bawah kami mentakrifkan dua alat carian penerbangan:
- **Primary** — meliputi Paris, Tokyo, dan Barcelona
- **Backup** — meliputi Berlin, Sydney, dan New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Ejen Merenung Diri dengan Pemulihan Ralat

Ejen di bawah ini diarahkan untuk mencuba sistem penerbangan utama terlebih dahulu, mengenal pasti kegagalan, dan secara telus beralih semula kepada sistem sandaran. Selepas setiap respons ia secara ringkas menilai diri sama ada ia telah menjawab sepenuhnya soalan pengguna.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Corak Penilaian Kendiri

Satu lagi aspek metakognisi ialah **penilaian kendiri**: ejen berasingan (atau ejen yang sama dalam laluan kedua) menyemak sesuatu jawapan dari segi kelengkapan, ketepatan, dan kebermanfaatan.

Di bawah kami mencipta ejen `ResponseEvaluator` yang memberi skor kepada jawapan ejen pelancongan berdasarkan tiga dimensi.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Ringkasan

In this lesson you learned how to build **ejen metakognitif** using the Microsoft Agent Framework:

- **Refleksi kendiri**: Ejen yang memantau penalaran mereka sendiri dan berkomunikasi secara telus apa yang berlaku.
- **Pemulihan ralat dengan sandaran**: Satu pola alat utama + sandaran di mana ejen mengesan kegagalan (contohnya, ralat 404) dan secara automatik mencuba sumber alternatif.
- **Penilaian kendiri**: Satu ejen penilai berasingan yang memberi skor kepada respons dari segi kelengkapan, ketepatan, dan kebermanfaatan.

Corak-corak ini menjadikan ejen lebih teguh, telus, dan boleh dipercayai — kualiti penting untuk penyebaran pengeluaran.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi ralat atau ketidaktepatan. Dokumen asal dalam bahasa asalnya hendaklah dianggap sebagai sumber rujukan yang muktamad. Untuk maklumat penting, disyorkan mendapatkan terjemahan profesional oleh penterjemah manusia. Kami tidak bertanggungjawab terhadap sebarang salah faham atau penafsiran yang salah yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
